# Stage1 v5 连续归因金额标签恒等式核验

冻结范围：只读 2018–2020，保留 2021–2022；0 拟合、0 新账户、0 新 Recorder。先核对 Qlib 来源和 NAS 绑定，再检查现金与 SOXL 选择对数分量是否只由开盘权重和未来开盘收益确定。

In [1]:
import os, json, hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd(); card=root/"research/stage1_v5_continuous_component_identity_20261030"
plan=json.loads((card/"plan.json").read_text())
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["budget"]=={"new_model_fits":0,"new_native_accounts":0,"new_recorders":0,"new_market_requests":0}
assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
for spec in plan["sources"].values():
    source=root/spec["card"]
    assert sha(source/"summary.json")==spec["summary_sha256"]
    assert sha(source/"verification_manifest.json")==spec["manifest_sha256"]
    assert json.loads((source/"nas_binding.json").read_text())["evidence"]["sha256"]==spec["nas_archive_sha256"]
qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)

def load(role,name):
    spec=plan["sources"][role]
    found,receipt=load_verified_recorder_objects(root/spec["card"],expected_recorder_id=spec["recorder_id"],artifacts=[name],expected_manifest_sha256=spec["manifest_sha256"])
    assert receipt is not None
    return found[name]
daily=load("components","analysis/daily")
frame=load("frame","inputs/covered_frame")
book=load("parent","analysis/native/dated_f696_0")
path=book["independent_replay"]
assert len(daily)==len(frame)==len(path)==1056
assert pd.to_datetime(daily.trade_date).reset_index(drop=True).equals(pd.to_datetime(frame.trade_date).reset_index(drop=True))
assert pd.to_datetime(daily.trade_date).reset_index(drop=True).equals(pd.to_datetime(path.trade_date).reset_index(drop=True))
assert book["audit"]["status"]=="passed" and book["audit"]["slippage_bps"]==0
print("Qlib and NAS source identity passed; 1056 aligned source rows")

[18685:MainThread](2026-10-30 13:07:17,800) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[18685:MainThread](2026-10-30 13:07:18,161) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[18685:MainThread](2026-10-30 13:07:18,161) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Qlib and NAS source identity passed; 1056 aligned source rows


In [2]:
# Only discovery outcomes are touched below. The frame retains reserved years in memory but no outcome math uses them.
mask=pd.to_datetime(daily.trade_date).dt.year.isin(plan["discovery_years"]).to_numpy()
d=daily.loc[mask].reset_index(drop=True)
f=frame.loc[mask].reset_index(drop=True)
b=path.loc[mask].reset_index(drop=True)
assert len(d)==558 and set(d.year)==set(plan["discovery_years"])
assert (pd.to_datetime(f.feature_available_at)<pd.to_datetime(f.decision_at)).all()
r_b=f.soxx_exit_open.to_numpy(float)/f.soxx_open.to_numpy(float)-1
r_x=f.soxl_exit_open.to_numpy(float)/f.soxl_open.to_numpy(float)-1
r_p=b.strategy_return.to_numpy(float)
paired=np.log1p(r_p)-np.log1p(r_b)
relative=r_p-r_b
scale=np.divide(paired,relative,out=1/(1+r_b),where=np.abs(relative)>1e-12)
cash_pred=-d.opening_cash_weight.to_numpy(float)*r_b*scale
selection_pred=d.opening_soxl_weight.to_numpy(float)*(r_x-r_b)*scale
cash_err=np.abs(d.cash_log.to_numpy(float)-cash_pred)
sel_err=np.abs(d.soxl_selection_log.to_numpy(float)-selection_pred)
nonrecovery=~d.recovery_action.to_numpy(bool)
assert np.isfinite(cash_err).all() and np.isfinite(sel_err).all()
assert cash_err.max()<plan["tolerance"]
assert sel_err[nonrecovery].max()<plan["tolerance"]
annual=[]
for year in plan["discovery_years"]:
    y=d.year.to_numpy()==year
    annual.append({"year":year,"sessions":int(y.sum()),"recovery_sessions":int((y&~nonrecovery).sum()),"cash_max_abs_error":float(cash_err[y].max()),"selection_nonrecovery_max_abs_error":float(sel_err[y&nonrecovery].max()),"cash_component_sum":float(d.cash_log.to_numpy(float)[y].sum()),"selection_component_sum":float(d.soxl_selection_log.to_numpy(float)[y].sum()),"relative_log_excess_sum":float(d.paired_log_excess.to_numpy(float)[y].sum())})
result={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"source_recorder_ids":{k:v["recorder_id"] for k,v in plan["sources"].items()},"discovery_sessions":len(d),"reserved_years_untouched":plan["reserved_years"],"cash_max_abs_error":float(cash_err.max()),"selection_nonrecovery_max_abs_error":float(sel_err[nonrecovery].max()),"recovery_days":int((~nonrecovery).sum()),"selection_recovery_max_abs_difference_from_static_formula":float(sel_err[~nonrecovery].max()) if (~nonrecovery).any() else 0.0,"cash_nonzero_weight_days":int((np.abs(d.opening_cash_weight.to_numpy(float))>1e-12).sum()),"soxl_nonzero_weight_nonrecovery_days":int(((np.abs(d.opening_soxl_weight.to_numpy(float))>1e-12)&nonrecovery).sum()),"annual":annual,"fits":0,"new_accounts":0,"decision":"continuous_component_amounts_are_exposure_scaled_future_price_returns_on_nonrecovery_days"}
(card/"audit.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
pd.DataFrame(annual).to_csv(card/"annual_identity.csv",index=False)
print(json.dumps(result,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_continuous_component_identity_20261030",
  "plan_sha256": "59bd5fbebc03c6f11ee9610d599526313f43974da0343b3bd35be65571e0da95",
  "source_recorder_ids": {
    "components": "240945b6fe474524b4feff43ad7f2b6e",
    "frame": "748eb4209e314fb6a6c41c19498d6d03",
    "parent": "2994f6afbfc54f279cefec362b220001"
  },
  "discovery_sessions": 558,
  "reserved_years_untouched": [
    2021,
    2022
  ],
  "cash_max_abs_error": 0.0,
  "selection_nonrecovery_max_abs_error": 3.122502256758253e-17,
  "recovery_days": 10,
  "selection_recovery_max_abs_difference_from_static_formula": 7.453889935837843e-20,
  "cash_nonzero_weight_days": 558,
  "soxl_nonzero_weight_nonrecovery_days": 548,
  "annual": [
    {
      "year": 2018,
      "sessions": 53,
      "recovery_sessions": 0,
      "cash_max_abs_error": 0.0,
      "selection_nonrecovery_max_abs_error": 2.6020852139652106e-17,
      "cash_component_sum": 0.04314457121426784,
      "selection_component_sum": -0.0475791744016

## 中文结论

SOXX 硬门：没有新候选账户，0/2/5bp 与独立前瞻检验均未运行，`historical_research_passed=false`。本卡只读已备份并核验的 Qlib 调整价 0bp Tiered 逐层点时父本；2018–2020 的 558 个发现期交易日，现金归因金额等于 `−开盘现金权重×SOXX 次开盘收益×共同对数尺度`，最大误差 0；SOXL 选择金额等于 `开盘 SOXL 权重×(SOXL−SOXX 次开盘收益)×尺度`，最大误差 3.13e−17。10 个 2020 恢复日该选择等式也成立；恢复及交易时点差另在残差中，不应混入本标签。

2018 年 53 日现金贡献 +0.04314、选择 −0.04758、总相对对数 −0.00497；2019 年 252 日现金 −0.13335、选择 +0.21787、总 +0.08322；2020 年 253 日现金 −0.07292、选择 +0.08259、总 +0.03602。先前已核验的最弱 126 日（2020-03-19 至 09-17）现金 −0.24045，最强 126 日（2022-03-31 至 09-30）现金 +0.28019；后者只是父本归因背景，2021/2022 目标值未用于本恒等式计算或特征选择。

原因：这两个连续金额是**既有开盘仓位对未来价格收益的代数缩放**。换成金额目标不会产生新的决策前信息；以相同原39项或普通分钟收益再次拟合，不构成新的经济机制。此判断不否定更长持仓路径、自身风险反馈或不同决策前来源，但它们必须另立动作标签与点时信息门。此卡无预测、目标改动、整股订单、费用变化或新 NAV，不能声称超额。下一步只读核查尚未测试的点时信息与已关闭路线的重叠，再决定是否有可冻结的新机制；若无，记录 `needs_distinct_decision_time_mechanism`。